# Questionnaires Preprocessing

Computes FSS2 dimension scores, IMI, RSME, and frustration from raw Qualtrics exports.
Produces a unified long-format CSV per paradigm.

**Sources (in `questionnaires/manually_cleaned/`):**
- `EEG_Flow_Active_questionnaires_cleaned.xlsx` — wide format, 4 blocks per row
- `EEG_Flow_Passive_questionnaires_cleaned.xlsx` — long format, one row per block

**Output on NAS:**
- `{date}_active_questionnaires.csv`
- `{date}_passive_questionnaires.csv`

**Format:** `participant | group | block | condition | task | measure | score`


In [ ]:
from datetime import datetime
from pathlib import Path
import sys
import numpy as np
import pandas as pd

DATE_STR    = datetime.today().strftime("%Y%m%d")
CLEANED_ACTIVE_DIR =  Path("L:/EEG_Flow_data/Study_EEG_Flow_Active/raw/participant_records/questionnaires/manually_cleaned")
CLEANED_PASSIVE_DIR =  Path("L:/EEG_Flow_data/Study_EEG_Flow_Passive/raw/participant_records/questionnaires/manually_cleaned")


if sys.platform == "win32":
    OUT_ACTIVE  = Path("L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/outputs/measures/behavior")
    OUT_PASSIVE = Path("L:/EEG_Flow_data/Study_EEG_Flow_Passive/derivatives_v2/outputs/measures/behavior")
else:
    OUT_ACTIVE  = Path("/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/measures/behavior")
    OUT_PASSIVE = Path("/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Passive/derivatives_v2/measures/behavior")

OUT_ACTIVE.mkdir(parents=True, exist_ok=True)
OUT_PASSIVE.mkdir(parents=True, exist_ok=True)

def map_condition(group, block):
    odd_groups = {"G1", "G3", "G5", "G7"}
    if block == 1: return "AudioOnly1"
    if block == 4: return "AudioOnly2"
    if group in odd_groups:
        return "Low Flow" if block == 2 else "High Flow"
    else:
        return "High Flow" if block == 2 else "Low Flow"

def map_task(block):
    return "oddball_only" if block in [1, 4] else "oddball_game"

print("Config OK")


In [ ]:
# ==============================================================================
# FSS2 definitions and shared functions
# ==============================================================================

FSS2_DIMENSIONS = [
    (1, "CSB", [1, 10, 19, 28]),
    (2, "MAA", [2, 11, 20, 29]),
    (3, "CC",  [3, 12, 21, 30]),
    (4, "UF",  [4, 13, 22, 31]),
    (5, "CTH", [5, 14, 23, 32]),
    (6, "SC",  [6, 15, 24, 33]),
    (7, "LSC", [7, 16, 25, 34]),
    (8, "TT",  [8, 17, 26, 35]),
    (9, "AE",  [9, 18, 27, 36]),
]
AGG6_DIMS  = {"MAA", "CTH", "SC", "LSC", "TT", "AE"}
AGG3_DIMS  = {"CSB", "CC", "UF"}
MISSING    = -99


def safe_mean(values):
    clean = [v for v in values if not pd.isna(v) and v != MISSING]
    return float(np.mean(clean)) if clean else np.nan


def compute_fss2(row, block, prefix="Q_FSS2"):
    scores = {}
    for dim_num, dim_name, q_list in FSS2_DIMENSIONS:
        vals = [row.get(f"{prefix}_{block}_{q}", np.nan) for q in q_list]
        scores[f"FSS2_{dim_num}_{dim_name}"] = safe_mean(vals)
    dim_vals = [scores[f"FSS2_{d}_{n}"] for d, n, _ in FSS2_DIMENSIONS]
    scores["FSS2_Global"] = safe_mean(dim_vals)
    scores["FSS2_Agg3"]   = safe_mean([scores[f"FSS2_{d}_{n}"] for d, n, _ in FSS2_DIMENSIONS if n in AGG3_DIMS])
    scores["FSS2_Agg6"]   = safe_mean([scores[f"FSS2_{d}_{n}"] for d, n, _ in FSS2_DIMENSIONS if n in AGG6_DIMS])
    return scores


def compute_imi(row, block):
    return safe_mean([row.get(f"Q_IMI_{block}_{q}", np.nan) for q in [1, 2, 3, 4]])


def to_long(participant, group, block, scores):
    condition = map_condition(group, block)
    task      = map_task(block)
    return [{"participant": participant, "group": group, "block": block,
             "condition": condition, "task": task, "measure": m, "score": s}
            for m, s in scores.items()]

print("Functions ready.")


## Active — wide to long

In [ ]:
df_act = pd.read_excel(CLEANED_ACTIVE_DIR / "EEG_Flow_Active_questionnaires_manuallycleaned_wide_1row_per_participant.xlsx")
df_act = df_act.rename(columns={"Q_token": "participant"})
print(f"Active: {len(df_act)} rows | {df_act['participant'].nunique()} participants")

rows = []
for _, row in df_act.iterrows():
    p, g = row["participant"], row["Group"]
    for block in [1, 2, 3, 4]:
        s = {}
        s.update(compute_fss2(row, block, prefix="Q_FSS2"))
        s["IMI"]   = compute_imi(row, block)
        # Handle RSME typo in source (Q_RSME_1 vs Q_RMSE_2/3/4)
        rsme_key   = f"Q_RSME_{block}" if f"Q_RSME_{block}" in row.index else f"Q_RMSE_{block}"
        rsme_val   = row.get(rsme_key, np.nan)
        s["RSME"]  = np.nan if rsme_val == MISSING else rsme_val
        frust_val  = row.get(f"Q_frust_{block}", np.nan)
        s["frust"] = np.nan if frust_val == MISSING else frust_val
        rows.extend(to_long(p, g, block, s))

df_active_long = pd.DataFrame(rows)
print(f"Active long: {len(df_active_long)} rows")
df_active_long.head(6)


## Passive — long to long

In [ ]:
df_pas = pd.read_excel(CLEANED_PASSIVE_DIR / "EEG_Flow_Passive_questionnaires_manuallycleaned_wide_1row_per_block.xlsx")
df_pas = df_pas.rename(columns={"Q_token": "participant"})
print(f"Passive: {len(df_pas)} rows | {df_pas['participant'].nunique()} participants")
print(f"Columns sample: {df_pas.columns[:10].tolist()}")

rows = []
for _, row in df_pas.iterrows():
    p, g, block = row["participant"], row["Group"], int(row["Q_Block"])
    s = {}
    # Passive uses Q_FSS_{q} without block number
    for dim_num, dim_name, q_list in FSS2_DIMENSIONS:
        vals = [row.get(f"Q_FSS_{q}", np.nan) for q in q_list]
        s[f"FSS_{dim_num}_{dim_name}"] = safe_mean(vals)
    dim_vals = [s[f"FSS_{d}_{n}"] for d, n, _ in FSS2_DIMENSIONS]
    s["FSS_Global"] = safe_mean(dim_vals)
    s["FSS_Agg3"]   = safe_mean([s[f"FSS_{d}_{n}"] for d, n, _ in FSS2_DIMENSIONS if n in AGG3_DIMS])
    s["FSS_Agg6"]   = safe_mean([s[f"FSS_{d}_{n}"] for d, n, _ in FSS2_DIMENSIONS if n in AGG6_DIMS])
    # IMI, RSME, frust — adapt column names to your passive source
    s["IMI"]   = safe_mean([row.get(f"Q_IMI_{q}", np.nan) for q in [1, 2, 3, 4]])
    s["RSME"]  = row.get("Q_RSME_1", np.nan)
    s["frust"] = row.get("Q_frust",  np.nan)
    rows.extend(to_long(p, g, block, s))

df_passive_long = pd.DataFrame(rows)
print(f"Passive long: {len(df_passive_long)} rows")
df_passive_long.head(6)


## Export to NAS

In [ ]:
n_act = df_active_long["participant"].nunique()
n_pas = df_passive_long["participant"].nunique()

fname_act = OUT_ACTIVE  / f"{DATE_STR}_active_questionnaires.csv"
fname_pas = OUT_PASSIVE / f"{DATE_STR}_passive_questionnaires.csv"

df_active_long.to_csv(fname_act,  index=False)
df_passive_long.to_csv(fname_pas, index=False)

print(f"Saved active  ({n_act}p)  -> {fname_act.name}")
print(f"Saved passive ({n_pas}p) -> {fname_pas.name}")
